# Answer using only the available notes

Imagine three shop notes: one about shipping, one about returns and one about account passwords. When someone asks about shipping, we want to find the shipping note and show its ID in the answer. When no note covers a question, we should say we do not know.

Before running: which note should the program retrieve for "How long does shipping take?" What about "astronomy quasar"? The code below turns notes and questions into numeric word counts and compares them. This is an offline search baseline, **not** a language model that writes new facts. See [the retrieval lesson](../../docs/06-ai-applications.md).

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
documents = {'shipping': 'Shipping takes three business days.', 'returns': 'Returns are accepted within 30 days.', 'security': 'Account passwords must not be shared.'}
ids = list(documents)
index = TfidfVectorizer(stop_words='english').fit(list(documents.values()))
vectors = index.transform(list(documents.values()))
def retrieve(question, threshold=0.1):
    scores = cosine_similarity(index.transform([question]), vectors)[0]
    best = scores.argmax()
    if scores[best] <= threshold:
        return None
    return ids[best], documents[ids[best]], float(scores[best])
def respond(question):
    match = retrieve(question)
    return 'I do not know from these documents.' if match is None else f'{match[1]} [source: {match[0]}]'
assert retrieve('How long does shipping take?')[0] == 'shipping'
assert retrieve('astronomy quasar') is None
assert 'returns' in respond('Are returns allowed?')
print(respond('How long does shipping take?'))
print(respond('astronomy quasar'))

## Check
This is a retrieval baseline, not generative RAG. Add a paraphrase that TF-IDF misses; a measured gap is a reason to try an embedding model. The source text must never be promoted to tool instructions.